In [3]:
import os, json, uuid, httpx
from pathlib import Path
from datetime import datetime
 
import pytesseract
import cv2
import numpy as np
from PIL import Image

In [4]:
try:
    from pdf2image import convert_from_path
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    print("⚠️  pdf2image not installed — PDF support disabled. Install poppler for Windows.")

In [5]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [6]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:3b"
OUTPUT_DIR   = Path("ocr_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [7]:
def load_document(file_path: str) -> list:
    path = Path(file_path)
    ext  = path.suffix.lower()
 
    if ext == ".pdf":
        if not PDF_SUPPORT:
            raise RuntimeError("pdf2image not available. Install poppler.")
        # poppler_path needed on Windows — adjust if needed
        # Download from: https://github.com/oschwartz10612/poppler-windows/releases
        # Then set: poppler_path=r"C:\poppler\Library\bin"
        pages = convert_from_path(str(path), dpi=300, poppler_path=r"C:\poppler\poppler-25.12.0\Library\bin")
        print(f"  📄 PDF loaded: {len(pages)} page(s)")
        return pages
 
    elif ext in [".png", ".jpg", ".jpeg", ".tiff", ".bmp"]:
        img = Image.open(path).convert("RGB")
        print(f"  🖼️  Image loaded: {img.size}")
        return [img]
 
    else:
        raise ValueError(f"Unsupported file type: {ext}")
 

In [8]:
def preprocess(pil_img: Image.Image) -> Image.Image:
    """
    Grayscale → Denoise → Adaptive threshold
    Makes text sharp and clean for Tesseract
    """
    # PIL → OpenCV
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray   = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
 
    # Denoise
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
 
    # Adaptive threshold — handles uneven lighting in scans
    thresh = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
 
    return Image.fromarray(thresh)
 

In [9]:
def run_ocr(pages: list) -> str:
    """Runs Tesseract on all pages, returns combined raw text."""
    config = "--oem 3 --psm 6"
    texts  = []
 
    for i, page in enumerate(pages):
        processed = preprocess(page)
        text      = pytesseract.image_to_string(processed, config=config)
        texts.append(text)
        print(f"  📝 Page {i+1}: {len(text)} chars extracted")
 
    return "\n".join(texts)
 

In [10]:
PROMPTS = {
 
    "PAN_CARD": """
Extract fields from this Company PAN Card OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "pan_number":    "10-char PAN e.g. AAKCM1234C",
  "entity_name":   "Full company name",
  "date_of_reg":   "DD/MM/YYYY",
  "entity_type":   "COMPANY or INDIVIDUAL",
  "issuing_auth":  "Income Tax Department"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "GST_CERTIFICATE": """
Extract fields from this GST Registration Certificate OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "gstin":              "15-char GSTIN",
  "pan_number":         "10-char PAN",
  "legal_name":         "Legal name of business",
  "trade_name":         "Trade name if different",
  "state_code":         "2-digit state code",
  "state":              "State name",
  "status":             "ACTIVE or CANCELLED",
  "registration_date":  "DD/MM/YYYY",
  "constitution":       "Private Limited etc",
  "address":            "Full address",
  "pincode":            "6-digit PIN"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "LEI_CERTIFICATE": """
Extract fields from this LEI Certificate OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "lei_code":           "20-char LEI",
  "legal_name":         "Legal name",
  "cin":                "CIN number",
  "pan_number":         "PAN number",
  "status":             "ACTIVE or INACTIVE",
  "registration_date":  "DD/MM/YYYY",
  "renewal_date":       "DD/MM/YYYY",
  "issuing_lou":        "Issuing organization",
  "country":            "Country"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "INCORPORATION_CERTIFICATE": """
Extract fields from this Certificate of Incorporation OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "pan_number":         "PAN number",
  "date_of_incorp":     "DD/MM/YYYY",
  "company_type":       "Private Limited etc",
  "authorized_capital": "Amount in numbers only e.g. 2500000",
  "state":              "State name",
  "roc":                "ROC office",
  "address":            "Registered address",
  "pincode":            "6-digit PIN"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "MOA": """
Extract fields from this Memorandum of Association OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "state":              "State of registered office",
  "address":            "Registered address",
  "pincode":            "6-digit PIN",
  "authorized_capital": "Amount in numbers only",
  "main_objects":       ["list of main business objects"],
  "subscribers": [
    {"name": "subscriber name", "shares": "number of shares"}
  ]
}
 
OCR TEXT:
{ocr_text}
""",
 
    "AOA": """
Extract fields from this Articles of Association OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "authorized_capital": "Amount in numbers only",
  "min_directors":      "minimum number",
  "max_directors":      "maximum number",
  "directors": [
    {"name": "director name", "din": "DIN number"}
  ]
}
 
OCR TEXT:
{ocr_text}
""",
 
    "REGISTERED_ADDRESS": """
Extract fields from this Registered Address / INC-22 OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":    "Full company name",
  "cin":             "CIN number",
  "pan_number":      "PAN number",
  "gstin":           "GSTIN",
  "lei":             "LEI code",
  "address_line1":   "Building/flat",
  "address_line2":   "Street/road",
  "area":            "Area/locality",
  "city":            "City",
  "state":           "State",
  "pincode":         "6-digit PIN",
  "srn":             "Service request number",
  "filing_date":     "DD/MM/YYYY",
  "approval_date":   "DD/MM/YYYY",
  "status":          "APPROVED or PENDING"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "ELECTRICITY_BILL": """
Extract fields from this Electricity Bill OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "consumer_name":    "Name on bill",
  "consumer_number":  "Consumer/account number",
  "discom":           "Electricity provider name",
  "address":          "Consumer address",
  "pincode":          "6-digit PIN",
  "bill_number":      "Bill reference number",
  "bill_date":        "DD/MM/YYYY",
  "due_date":         "DD/MM/YYYY",
  "billing_period":   "Month Year",
  "units_consumed":   "Number only",
  "total_amount":     "Number only e.g. 4693.70",
  "connection_type":  "Commercial or Residential"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "TELEPHONE_BILL": """
Extract fields from this Telephone/Landline Bill OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "account_name":     "Name on bill",
  "account_number":   "Account number",
  "telephone_number": "Phone number",
  "provider":         "Telecom provider name",
  "address":          "Address on bill",
  "pincode":          "6-digit PIN",
  "bill_number":      "Bill reference number",
  "bill_date":        "DD/MM/YYYY",
  "due_date":         "DD/MM/YYYY",
  "billing_period":   "Month Year",
  "total_amount":     "Number only",
  "connection_type":  "Landline or Mobile"
}
 
OCR TEXT:
{ocr_text}
"""
}

In [11]:
def extract_with_llm(ocr_text: str, doc_type: str) -> dict:
    """Sends OCR text to Qwen3 via Ollama, returns parsed JSON."""
 
    prompt_template = PROMPTS.get(doc_type)
    if not prompt_template:
        raise ValueError(f"No prompt defined for doc_type: {doc_type}")
 
    prompt = prompt_template.replace("{ocr_text}", ocr_text)
 
    print(f"  🤖 Sending to Qwen3...")
 
    try:
        response = httpx.post(
            OLLAMA_URL,
            json={
                "model":  OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                # Tell Qwen3 to think less, just extract — faster
                "options": {
                    "temperature": 0.1,
                    "top_p": 0.9,
                }
            },
            timeout=120.0  # Qwen3 can be slow on CPU
        )
        response.raise_for_status()
        raw_response = response.json()["response"]
 
        # ── Clean up response ─────────────────────────────────
        # Qwen3 sometimes wraps in ```json ... ```
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        cleaned = cleaned.strip()
 
        # Also strip <think>...</think> blocks if present
        if "<think>" in cleaned:
            import re
            cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL).strip()
 
        parsed = json.loads(cleaned)
        print(f"  ✅ JSON extracted: {len(parsed)} fields")
        return parsed
 
    except httpx.ConnectError:
        print("  ❌ Ollama not running. Start with: ollama serve")
        return {"error": "OLLAMA_NOT_RUNNING"}
    except json.JSONDecodeError as e:
        print(f"  ⚠️  JSON parse failed: {e}")
        print(f"  Raw response: {raw_response[:200]}")
        return {"error": "JSON_PARSE_FAILED", "raw": raw_response[:500]}
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return {"error": str(e)}
 

In [12]:
def build_document_json(
    file_path: str,
    doc_type:  str,
    fields:    dict,
    ocr_text:  str
) -> dict:
    """Wraps extracted fields in the standard compliance JSON envelope."""
 
    # Cross-check fields — same keys across ALL documents
    # Used for consistency verification later
    cross_check = {
        "pan_number":   fields.get("pan_number")   or fields.get("pan"),
        "company_name": fields.get("entity_name")  or fields.get("legal_name")   or
                        fields.get("company_name") or fields.get("consumer_name") or
                        fields.get("account_name"),
        "address":      fields.get("address")      or
                        f"{fields.get('address_line1','')} {fields.get('address_line2','')}".strip(),
        "pincode":      fields.get("pincode"),
        "cin":          fields.get("cin"),
        "gstin":        fields.get("gstin"),
    }
 
    # Remove None values
    cross_check = {k: v for k, v in cross_check.items() if v}
 
    return {
        "doc_id":       str(uuid.uuid4())[:8].upper(),
        "doc_type":     doc_type,
        "source_file":  Path(file_path).name,
        "extracted_at": datetime.now().isoformat(),
        "fields":       fields,
        "cross_check":  cross_check,
        "ocr_raw_text": ocr_text[:1000],  # first 1000 chars for debug
        "status":       "FAILED" if "error" in fields else "EXTRACTED"
    }

In [13]:
def process_document(file_path: str, doc_type: str) -> dict:
    """Full pipeline for a single document."""
 
    print(f"\n{'─'*55}")
    print(f"  Processing: {Path(file_path).name}")
    print(f"  Doc Type  : {doc_type}")
    print(f"{'─'*55}")
 
    # Step 1 — Load
    pages = load_document(file_path)
 
    # Step 2+3 — Preprocess + OCR
    ocr_text = run_ocr(pages)
 
    # Step 4 — LLM extraction
    fields = extract_with_llm(ocr_text, doc_type)
 
    # Step 5 — Build final JSON
    doc_json = build_document_json(file_path, doc_type, fields, ocr_text)
 
    # Save to output folder
    out_file = OUTPUT_DIR / f"{doc_type}_{doc_json['doc_id']}.json"
    with open(out_file, "w") as f:
        json.dump(doc_json, f, indent=2)
 
    print(f"  💾 Saved: {out_file}")
    return doc_json

In [14]:
DOCUMENTS = [
    ("01_Company_PAN_Card.png",              "PAN_CARD"),
    ("02_GST_Certificate.pdf",               "GST_CERTIFICATE"),
    ("03_LEI_Certificate.pdf",               "LEI_CERTIFICATE"),
    ("04_Certificate_of_Incorporation.pdf",  "INCORPORATION_CERTIFICATE"),
    ("05_MOA.pdf",                           "MOA"),
    ("06_AOA.pdf",                           "AOA"),
    ("07_Registered_Address_INC22.pdf",      "REGISTERED_ADDRESS"),
    ("08_Electricity_Bill.pdf",              "ELECTRICITY_BILL"),
    ("09_Telephone_Landline_Bill.pdf",       "TELEPHONE_BILL"),
]
 


In [15]:
def normalize(value: str) -> str:
    """Normalize string for comparison — lowercase, strip extra spaces."""
    if not value:
        return ""
    return " ".join(str(value).lower().strip().split())

In [16]:
def run_cross_checks(results: list) -> dict:
    """
    Full cross-check engine.
    Checks 6 critical parameters across all documents that should contain them.
    Pinpoints exactly which document has the inconsistent value.
    """
 
    # Parameters to check → their key in cross_check block
    CHECKS = {
        "PAN Number":   "pan_number",
        "Company Name": "company_name",
        "Pincode":      "pincode",
        "CIN":          "cin",
        "GSTIN":        "gstin",
        "Address":      "address",
    }
 
    # Which doc types are expected to have each field
    FIELD_PRESENCE = {
        "pan_number": [
            "PAN_CARD", "GST_CERTIFICATE", "LEI_CERTIFICATE",
            "INCORPORATION_CERTIFICATE", "REGISTERED_ADDRESS"
        ],
        "company_name": [
            "PAN_CARD", "GST_CERTIFICATE", "LEI_CERTIFICATE",
            "INCORPORATION_CERTIFICATE", "MOA", "AOA",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
        "pincode": [
            "GST_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
        "cin": [
            "LEI_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "MOA", "AOA", "REGISTERED_ADDRESS"
        ],
        "gstin": [
            "GST_CERTIFICATE", "REGISTERED_ADDRESS"
        ],
        "address": [
            "GST_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
    }
 
    report = {"passed": [], "failed": [], "warnings": []}
 
    print("\n" + "=" * 65)
    print("   CROSS-CHECK REPORT")
    print("=" * 65)
 
    for label, key in CHECKS.items():
        expected_docs = FIELD_PRESENCE.get(key, [])
 
        found        = {}  # doc_type -> raw value
        missing_from = []
 
        for r in results:
            if r["doc_type"] not in expected_docs:
                continue
            val = r["cross_check"].get(key)
            if val:
                found[r["doc_type"]] = val
            else:
                missing_from.append(r["doc_type"])
 
        print(f"\n  {'-' * 60}")
        print(f"  Checking : {label}")
 
        if not found:
            msg = "Not extracted from any document — possible OCR failure"
            print(f"  WARNING  : {msg}")
            report["warnings"].append({"parameter": label, "issue": msg})
            continue
 
        normalized  = {doc: normalize(val) for doc, val in found.items()}
        unique_vals = set(normalized.values())
 
        if len(unique_vals) == 1:
            print(f"  RESULT   : CONSISTENT  ->  {list(found.values())[0]}")
            for doc in found:
                print(f"     {doc:<44} OK")
            report["passed"].append({
                "parameter": label,
                "value":     list(found.values())[0],
                "docs":      list(found.keys())
            })
 
        else:
            print(f"  RESULT   : MISMATCH DETECTED")
 
            # Group docs by their normalized value
            value_groups = {}
            for doc, val in found.items():
                norm = normalize(val)
                value_groups.setdefault(norm, []).append((doc, val))
 
            # Majority = most docs agree on this value
            majority_norm = max(value_groups, key=lambda x: len(value_groups[x]))
            majority_val  = value_groups[majority_norm][0][1]
 
            for norm_val, doc_list in value_groups.items():
                is_majority = (norm_val == majority_norm)
                tag = "majority (expected)" if is_majority else "INCONSISTENT"
                for doc, raw_val in doc_list:
                    print(f"     {doc:<44} {tag}")
                    print(f"       Value: \"{raw_val}\"")
 
            inconsistent_docs = [
                doc
                for norm_val, doc_list in value_groups.items()
                if norm_val != majority_norm
                for doc, _ in doc_list
            ]
            inconsistent_vals = [
                val
                for norm_val, doc_list in value_groups.items()
                if norm_val != majority_norm
                for _, val in doc_list
            ]
 
            print(f"\n  ACTION REQUIRED:")
            print(f"    Kindly check the following document(s) : {inconsistent_docs}")
            print(f"    Check consistency for                  : [{label}]")
            print(f"    Inconsistent value(s) found            : {inconsistent_vals}")
            print(f"    Expected value (majority)              : \"{majority_val}\"")
 
            report["failed"].append({
                "parameter":           label,
                "majority_value":      majority_val,
                "inconsistent_docs":   inconsistent_docs,
                "inconsistent_values": inconsistent_vals,
                "all_values":          found
            })
 
        if missing_from:
            print(f"  WARNING  : Field not extracted from (possible OCR issue): {missing_from}")
            report["warnings"].append({
                "parameter":    label,
                "missing_from": missing_from,
                "issue":        "Expected but not extracted — verify manually"
            })
 
    # Final verdict
    print(f"\n  {'=' * 65}")
    print(f"  FINAL VERDICT")
    print(f"  {'=' * 65}")
    print(f"  Passed   : {len(report['passed'])} parameter(s)")
    print(f"  Failed   : {len(report['failed'])} parameter(s)")
    print(f"  Warnings : {len(report['warnings'])} parameter(s)")
 
    if report["failed"]:
        print(f"\n  INCONSISTENCIES FOUND — Review before proceeding:")
        for item in report["failed"]:
            print(f"    [{item['parameter']}]  ->  inconsistent in: {item['inconsistent_docs']}")
        print(f"\n  Status: COMPLIANCE FAILED — Manual review required")
    elif report["warnings"]:
        print(f"\n  Status: PARTIAL — Some fields missing, verify manually")
    else:
        print(f"\n  Status: ALL CHECKS PASSED — Documents are consistent")
 
    print(f"  {'=' * 65}")
    return report
 


In [17]:
def run_all():
    print("\n" + "=" * 65)
    print("   NBFC OCR PIPELINE — Processing all documents")
    print("=" * 65)
 
    results = []
    failed  = []
 
    for filename, doc_type in DOCUMENTS:
        if not Path(filename).exists():
            print(f"\n  File not found: {filename} — skipping")
            failed.append(filename)
            continue
        result = process_document(filename, doc_type)
        results.append(result)
 
    # Processing summary
    print("\n" + "=" * 65)
    print("   PROCESSING SUMMARY")
    print("=" * 65)
    print(f"  Total Documents : {len(DOCUMENTS)}")
    print(f"  Processed       : {len(results)}")
    print(f"  Not Found       : {len(failed)}")
 
    extracted = [r for r in results if r["status"] == "EXTRACTED"]
    errors    = [r for r in results if r["status"] == "FAILED"]
    print(f"  Extracted OK    : {len(extracted)}")
    print(f"  LLM Errors      : {len(errors)}")
 
    if errors:
        print(f"\n  Documents with extraction errors:")
        for e in errors:
            print(f"    {e['doc_type']} ({e['source_file']})")
 
    # Run full cross-check
    cross_check_report = run_cross_checks(results)
 
    # Save both outputs
    master_file = OUTPUT_DIR / "all_documents.json"
    with open(master_file, "w") as f:
        json.dump(results, f, indent=2)
 
    report_file = OUTPUT_DIR / "cross_check_report.json"
    with open(report_file, "w") as f:
        json.dump(cross_check_report, f, indent=2)
 
    print(f"\n  Master JSON    : {master_file}")
    print(f"  Cross-Check    : {report_file}")
    print("=" * 65)
 
    return results, cross_check_report
 

In [18]:
if __name__ == "__main__":
    run_all()
 


   NBFC OCR PIPELINE — Processing all documents

───────────────────────────────────────────────────────
  Processing: 01_Company_PAN_Card.png
  Doc Type  : PAN_CARD
───────────────────────────────────────────────────────
  🖼️  Image loaded: (1012, 638)
  📝 Page 1: 278 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSON extracted: 5 fields
  💾 Saved: ocr_output\PAN_CARD_29BE211D.json

───────────────────────────────────────────────────────
  Processing: 02_GST_Certificate.pdf
  Doc Type  : GST_CERTIFICATE
───────────────────────────────────────────────────────
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 1184 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSON extracted: 11 fields
  💾 Saved: ocr_output\GST_CERTIFICATE_78C331B7.json

───────────────────────────────────────────────────────
  Processing: 03_LEI_Certificate.pdf
  Doc Type  : LEI_CERTIFICATE
───────────────────────────────────────────────────────
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 1073 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSO